# Really Big Bank – Proof of Concept (BUSA3022 Assessment 2)

This notebook runs the thin working slices behind the Analytics Implementation Plan.

| Section | Proves | Report figure |
| --- | --- | --- |
| 1. Setup | | |
| 2. Part 1 ingest | Both sources load end to end and no record is silently dropped | Figure 2 |
| 3. Re-run | Loads are idempotent (re-running adds no duplicates) | |
| 4. Failure demo | A timeout is retried, then the last good load is kept and flagged stale | Presentation demo |
| 5. Part 2 metrics | Sources aligned on a Sydney trading-day axis, metrics match a hand calculation | Figure 3 |

Run the cells from top to bottom.

## 1. Setup

**Before running:** click the key icon in Colab's left sidebar, add a secret called `FRED_API_KEY` (free key from fred.stlouisfed.org) and turn on notebook access.

Paste your GitHub repository URL into `REPO_URL` below.

In [ ]:
REPO_URL = "https://github.com/your-username/your-repo.git"   # <- change this

import os, sys, subprocess
from pathlib import Path

if not Path("src/pipeline.py").exists():          # in Colab, fetch the code from GitHub
    repo_dir = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not Path(repo_dir).exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(repo_dir)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=False)
sys.path.insert(0, "src")
print("Working folder:", os.getcwd())

In [ ]:
try:
    from google.colab import userdata          # Colab secrets
    FRED_API_KEY = userdata.get("FRED_API_KEY")
except Exception:
    FRED_API_KEY = os.environ.get("FRED_API_KEY")
if not FRED_API_KEY:
    from getpass import getpass
    FRED_API_KEY = getpass("Paste your FRED API key: ")
print("FRED key loaded:", bool(FRED_API_KEY))

## 2. Part 1 – Ingest both sources (Figure 2)

The first run backfills about 200 days of history, so there are enough trading days for the 90-day dashboard plus the 20-day rolling window.

Raw responses go to `data/raw` untouched. Every row then passes through the quality rules. Bad rows go to `data/logs/reject_log.csv` with the rule they broke.
The `reconciles` column confirms that **rows_in = rows_passed + rows_already_loaded + rows_rejected**, so nothing went missing.

In [ ]:
import pandas as pd
from datetime import date, timedelta
import pipeline, metrics

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)   # plain numbers, no scientific notation
summary = pipeline.run(FRED_API_KEY, start=date.today() - timedelta(days=200))
summary[["source", "status", "rows_in", "rows_passed", "rows_already_loaded", "rows_rejected", "reconciles"]]

In [ ]:
rejects = pd.read_csv(pipeline.REJECT_LOG)
print(rejects["rule_failed"].value_counts().to_string(), "\n")
rejects.tail(10)   # screenshot this together with the summary above for Figure 2

## 3. Re-run straight away – idempotency check

A second run should load **0 new rows**. The overlapping dates appear as `rows_already_loaded`, and the clean files keep exactly one row per date.

In [ ]:
summary2 = pipeline.run(FRED_API_KEY, start=date.today() - timedelta(days=200))
display(summary2[["source", "rows_in", "rows_passed", "rows_already_loaded", "rows_rejected", "reconciles"]])
for name in ["vix_daily", "asx_daily"]:
    df = pipeline.load_clean(name)
    print(f"{name}: {len(df)} rows, duplicate dates = {df['date'].duplicated().sum()}")

## 4. Failure scenario demo – API timeout

This simulates a provider that never responds. Delays are shortened to 1, 2 and 4 **seconds** for the demo (the real pipeline waits 1, 2 and 4 minutes).
Watch for four attempts, the alert, the stale flag, and the clean data left untouched.

In [ ]:
import requests

def provider_never_responds():
    raise requests.Timeout("Read timed out after 30 seconds")

before = len(pipeline.load_clean("vix_daily"))
result = pipeline.process_source("FRED VIXCLS (simulated outage)", "vix_daily", provider_never_responds,
                                 pipeline.save_raw_vix, pipeline.vix_records, ["vix_close"],
                                 run_id="demo_timeout", delays=[1, 2, 4])
after = len(pipeline.load_clean("vix_daily"))

display(pd.read_csv(pipeline.ATTEMPT_LOG).query("run_id == 'demo_timeout'"))
print("Status:", result["status"])
print("Stale flag:", pipeline.STALE_FLAG.read_text().strip())
print(f"Clean VIX rows before {before}, after {after}: last good load kept")
pipeline.STALE_FLAG.unlink()   # clear the demo flag

## 5. Part 2 – Align on one time axis and calculate the metrics (Figure 3)

Each ASX trading day (Sydney) is paired with the VIX close from the **previous US date**, because the US market closes after the ASX does.
The `vix_us_date` column shows exactly which VIX value each row used.

In [ ]:
serve = metrics.run()
print("Dashboard table rows:", len(serve), "| data as at:", serve["data_as_at"].iloc[0])
serve[["trade_date", "asx_close", "asx_volume", "vol_avg_20d", "volatility_14d_pct",
       "vix_us_date", "vix_close", "vix_stale"]].tail(8)

### Hand check (Figure 3)

The same metrics recalculated with plain Python, the way you would in Excel, and compared with the pipeline. Every row should say `True`.
For the strongest evidence, also open `data/clean/market_daily.csv` in Excel and check one date yourself with `=AVERAGE()` and `=STDEV.S()`.

In [ ]:
check = metrics.hand_check(pd.read_csv("data/clean/market_daily.csv", parse_dates=["trade_date"]))
check   # screenshot for Figure 3